# Amazon Scraper API Exploration

## Project
Amazon Germany Wireless Headphones Price and Competitor Intelligence

## Purpose
This notebook tests the Amazon Scraper API, examines the JSON response structure and saves raw Amazon Germany search data for the ETL pipeline.

In [2]:
import sys
import requests
import pandas as pd
import numpy as np
import sqlalchemy
import psycopg2
import matplotlib

print("Python version:", sys.version)
print("Pandas version:", pd.__version__)
print("Requests version:", requests.__version__)
print("Environment setup successful.")

Python version: 3.13.13 | packaged by Anaconda, Inc. | (main, Apr 14 2026, 06:12:50) [MSC v.1942 64 bit (AMD64)]
Pandas version: 3.0.3
Requests version: 2.34.2
Environment setup successful.


## Import the required libraries

In [3]:
import os
import json
from datetime import datetime
from pathlib import Path

import pandas as pd
import requests
from dotenv import load_dotenv

print("Libraries imported successfully.")

Libraries imported successfully.


## Define the project paths

In [4]:


PROJECT_ROOT = Path.cwd().parent
RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"
ENV_FILE = PROJECT_ROOT / ".env"

RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Raw data folder:", RAW_DATA_DIR)
print("Environment file exists:", ENV_FILE.exists())

Project root: C:\Users\srika\Documents\Srikanth_Amazon_BI_Project
Raw data folder: C:\Users\srika\Documents\Srikanth_Amazon_BI_Project\data\raw
Environment file exists: True


## Load the API key securely

In [5]:
load_dotenv(ENV_FILE)

API_KEY = os.getenv("AMAZON_SCRAPER_API_KEY")

if not API_KEY:
    raise ValueError(
        "Amazon Scraper API key was not found. "
        "Check the AMAZON_SCRAPER_API_KEY value in the .env file."
    )

print("API key loaded successfully.")
print("API key is hidden for security.")

API key loaded successfully.
API key is hidden for security.


## Prepare the API request

In [6]:
API_URL = "https://api.amazonscraperapi.com/api/v1/amazon/search"

params = {
    "query": "wireless headphones",
    "domain": "de",
    "sort_by": "best_match",
    "api_key": API_KEY
}

print("API request configured.")
print("Search query:", params["query"])
print("Amazon marketplace: amazon.de")

API request configured.
Search query: wireless headphones
Amazon marketplace: amazon.de


## Send the first API request

In [8]:
try:
    response = requests.get(
        API_URL,
        params=params,
        timeout=60
    )

    print("HTTP status code:", response.status_code)

    response.raise_for_status()

    api_data = response.json()

    print("API request completed successfully.")
    print("Response type:", type(api_data))

except requests.exceptions.Timeout:
    print("The API request timed out.")

except requests.exceptions.HTTPError as error:
    print("HTTP error:", error)
    print("Response preview:", response.text[:500])

except requests.exceptions.RequestException as error:
    print("Request error:", error)

except json.JSONDecodeError:
    print("The API returned a response that was not valid JSON.")
    print("Response preview:", response.text[:500])

HTTP status code: 200
API request completed successfully.
Response type: <class 'dict'>


## Inspect the JSON safely

In [9]:
print("Top-level JSON keys:")

if isinstance(api_data, dict):
    print(list(api_data.keys()))
else:
    print("The response is not a dictionary.")

Top-level JSON keys:
['page', 'products', 'html']


## Display a small preview

In [11]:
preview = json.dumps(api_data, indent=2, ensure_ascii=False)

print(preview[:3000])

{
  "page": 1,
  "products": [
    {
      "asin": "B0FSH42VZW",
      "title": "JBL Tune 730 BT Wireless Over-Ear-Kopfhörermit JBL Pure Bass Sound, Bis zu 76 Stunden Musikwiedergabe, Bluetooth 6.0, leichtem, faltbarem Design, Google Fast Pair, Microsoft Swift Pair,Schwarz",
      "url": "https://www.amazon.de/sspa/click?ie=UTF8&spc=MTo3MzE3NTIzMTI1MDM2MTE1OjE3ODE3MDA0NzQ6c3BfYXRmOjMwMDgwMzQ2NTQ1MDkzMjo6MDo6&url=%2FJBL-Over-Ear-Kopfh%25C3%25B6rermit-Musikwiedergabe-Bluetooth-faltbarem-Black%2Fdp%2FB0FSH42VZW%2Fref%3Dsr_1_1_sspa%3Fdib%3DeyJ2IjoiMSJ9.6bFymcQNAFwDinbk1Lz45Bv0tRYEFM2vgvb8mjd0V8iSD2biGxbjiJ5ckNzNWOXzhJrIIFGifwcLQfGY2N9LAZq_3pq-BthWysJKQs1sz8CJw0V_IkkkSEtG4Ev0o3gdwS9uYx3gVKr285TXnpwHdT_pj__UlZj0Y4KC-Porqh1RNzxgriSwIGuTMz0rOEie_thvKWYu-1HLR9-dJ3djTdN3NgTzH7rc9zhpT6cd6G4.8CM7y20NpJ9fSYQFaAG6oFLrf-eK50zqY5jGNxbRHt8%26dib_tag%3Dse%26keywords%3Dwireless%2Bheadphones%26qid%3D1781700474%26sr%3D8-1-spons%26aref%3DEqvqSWMZQZ%26sp_csd%3Dd2lkZ2V0TmFtZT1zcF9hdGY%26psc%3D1&aref=EqvqSWMZQ

## Save the raw JSON response

In [12]:
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

raw_json_file = RAW_DATA_DIR / f"amazon_wireless_headphones_de_{timestamp}.json"

with open(raw_json_file, "w", encoding="utf-8") as file:
    json.dump(api_data, file, indent=2, ensure_ascii=False)

print("Raw JSON saved successfully.")
print("Saved file:", raw_json_file)

Raw JSON saved successfully.
Saved file: C:\Users\srika\Documents\Srikanth_Amazon_BI_Project\data\raw\amazon_wireless_headphones_de_20260617_145203.json


## verify the saved file

In [13]:
print("File exists:", raw_json_file.exists())
print("File size in bytes:", raw_json_file.stat().st_size)

File exists: True
File size in bytes: 30773
